# Kaggle Launcher — E2VID Reconstruction + YOLO Training
**Last updated: 2026-06-19  v54**

This notebook is a thin wrapper that runs the two Python scripts on Kaggle GPU.

**Before running:**
1. Confirm these two datasets are attached as inputs:
   - `gennepy/fred-events-ami` — events + coordinates (now includes sequences 44–47, 146)
   - `gennepy/fred-scripts-ami` — reconstruct.py, train_yolo.py
   - `gennepy/fred-frames-all` — pre-reconstructed frames for sequences 84/85/201/124 (skips re-reconstruction)
2. Set Runtime → Accelerator → **GPU T4 x2** (or P100)
3. Run all cells top to bottom

**What this run does (Run 6):**
- Restores sequences 84/85/201/124 from `fred-frames-all` (no re-reconstruction)
- Reconstructs sequences 44/45/46/47/146 fresh from events (new principled split)
- sequence_47 uses `start_s=7.0` (residual noise burst detected at 5–7 s)
- PNGs converted to JPEG inline during reconstruction (background thread) — no disk accumulation
- Trains YOLOv8s on sequences 44/45/46/47 + 84/85/201/124 (val: **146**)
- Zips new reconstruction frames (44/45/46/47/146) for GUI download
- Saves weights, KPIs, curves

**After completion:**
```bash
bash scripts/sync_from_kaggle.sh
bash scripts/sync_from_kaggle.sh --frames-zip
docker compose build e2vid && docker compose up -d
```

## 1 · Configuration — edit this cell

In [ ]:
from pathlib import Path
import datetime

# ── Kaggle dataset paths ───────────────────────────────────────────────────────
AMI_INPUT        = Path('/kaggle/input/datasets/gennepy/fred-events-ami')
SCRIPTS_INPUT    = Path('/kaggle/input/datasets/gennepy/fred-scripts-ami')
AMI_WORK         = Path('/kaggle/working')
PREV_RECON_INPUT = None  # reconstruct fresh at events_per_pixel=0.05

# ── Run control ───────────────────────────────────────────────────────────────
RESUME        = False
SKIP_TRAINING = False

# ── Sequences ─────────────────────────────────────────────────────────────────
# Run 6: 5 sequences from fred-frames-all; train on 84/85/124/201, val on 127
SEQUENCES     = [
    'sequence_84', 'sequence_85', 'sequence_124', 'sequence_201', 'sequence_127',
]
VAL_SEQUENCES = ['sequence_127']

# ── Per-sequence start_s overrides ────────────────────────────────────────────
START_S_OVERRIDES = {}  # no per-sequence overrides
DEFAULT_START_S = 5.0

# ── Training parameters ───────────────────────────────────────────────────────
MODEL  = 'yolov8s.pt'
EPOCHS = 100
BATCH  = 16

# ── Reconstruction parameters ─────────────────────────────────────────────────
EVENTS_PER_PIXEL = 0.05
SMOKE_EVENTS     = None   # e.g. 100_000 for a quick smoke test (~10 frames)

# ── Sequences to zip for GUI download (new reconstructions only) ──────────────
ZIP_SEQUENCES = ['sequence_84', 'sequence_85', 'sequence_124', 'sequence_201', 'sequence_127']

# ── Derived paths ─────────────────────────────────────────────────────────────
SCRIPTS_DIR = SCRIPTS_INPUT
EVENTS_ROOT = AMI_INPUT / 'data' / 'processed'
RAW_ROOT    = AMI_INPUT / 'data' / 'raw'
RECON_ROOT  = AMI_WORK  / 'data' / 'processed'
WEIGHTS_OUT = AMI_WORK  / 'yolo_e2vid.pt'
RUNS_DIR    = AMI_WORK  / 'yolo_runs'
LOG_FILE    = AMI_WORK  / 'logs' / f'run_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
DATASET_DIR = AMI_WORK  / 'yolo_e2vid'
WORK_DIR    = Path('/tmp/ami_work')

WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

train_seqs = [s for s in SEQUENCES if s not in VAL_SEQUENCES]
print('Model         :', MODEL)
print('Train seqs    :', train_seqs)
print('Val seqs      :', VAL_SEQUENCES)
print('Epochs        :', EPOCHS, '  Batch:', BATCH)
print('Resume        :', RESUME)
print('Skip training :', SKIP_TRAINING)
print('start_s       :', DEFAULT_START_S, ' overrides:', START_S_OVERRIDES)
print('Smoke events  :', SMOKE_EVENTS or 'full run')

## 2 · Install dependencies

In [ ]:
!pip install -q h5py ultralytics==8.4.54 imageio scikit-image pandas matplotlib
import torch
print(f'PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU detected. Go to Settings → Accelerator → GPU T4 x2, then restart.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}  '
      f'({torch.cuda.get_device_properties(0).total_memory // 1024**2} MB)')

## 3 · Helpers

In [ ]:
import os, subprocess, sys, datetime, shutil

# Copy scripts from input dataset to local working dir
LOCAL_SCRIPTS = Path('/kaggle/working/scripts')
LOCAL_SCRIPTS.mkdir(exist_ok=True)
for script in ['reconstruct.py', 'train_yolo.py']:
    shutil.copy(SCRIPTS_DIR / script, LOCAL_SCRIPTS / script)
print(f'Scripts copied to {LOCAL_SCRIPTS}')

def run_streaming(cmd):
    """Run a command, stream output to notebook and append to log file."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    with open(LOG_FILE, 'a') as lf:
        for line in process.stdout:
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    process.wait()
    return process.returncode

def log(msg):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(LOG_FILE, 'a') as f:
        f.write(line + '\n')

log('Helpers ready.')

## 4 · Cleanup — remove stale output before each run

In [ ]:
import shutil

if RESUME:
    print('RESUME=True — skipping cleanup, keeping existing outputs.')
else:
    for seq in SEQUENCES:
        recon_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if recon_dir.exists():
            shutil.rmtree(recon_dir)
            print(f'Cleaned: {recon_dir}')
        else:
            print(f'Nothing to clean: {recon_dir}')

    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    print(f'Cleaned: {DATASET_DIR}')

    shutil.rmtree(WORK_DIR / 'rpg_e2vid', ignore_errors=True)
    print(f'Cleaned: {WORK_DIR / "rpg_e2vid"}')

In [ ]:
# ── Restore previous reconstructions so they are not re-run ─────────────────
import shutil

if PREV_RECON_INPUT is not None and PREV_RECON_INPUT.exists():
    log('Restoring previous reconstructions from ' + str(PREV_RECON_INPUT))
    log('  Contents: ' + str([p.name for p in sorted(PREV_RECON_INPUT.iterdir())[:10]]))
    for seq in SEQUENCES:
        candidates = [
            PREV_RECON_INPUT / 'processed' / seq / 'reconstruction_e2vid',
            PREV_RECON_INPUT / 'data' / 'processed' / seq / 'reconstruction_e2vid',
            PREV_RECON_INPUT / seq,
        ]
        src_dir = next((p for p in candidates if p.exists()), None)
        dst_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if src_dir is not None and not dst_dir.exists():
            dst_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src_dir, dst_dir)
            n = len(list(dst_dir.glob('frame_*')))
            log(f'  {seq}: restored {n} frames')
        elif dst_dir.exists():
            log(f'  {seq}: already in working dir, skipping copy')
        else:
            log(f'  {seq}: not found in prev recon input — will reconstruct')
else:
    log('No PREV_RECON_INPUT — all sequences will be reconstructed from scratch')

## 5 · Reconstruct e2vid frames

In [ ]:
log('=== Reconstruction started ===')

for seq in SEQUENCES:
    zip_path = EVENTS_ROOT / seq / 'events.zip'
    out_dir  = RECON_ROOT  / seq / 'reconstruction_e2vid'

    if out_dir.exists() and (any(out_dir.glob('frame_*.png')) or any(out_dir.glob('frame_*.jpg'))):
        log(f'{seq}: frames already exist — skipping reconstruction')
        continue

    start_s = START_S_OVERRIDES.get(seq, DEFAULT_START_S)
    log(f'=== Reconstructing {seq} (start_s={start_s}, events_per_pixel={EVENTS_PER_PIXEL}) ===')
    cmd = [
        sys.executable, str(LOCAL_SCRIPTS / 'reconstruct.py'),
        '--zip_path',         str(zip_path),
        '--out_dir',          str(out_dir),
        '--work_dir',         str(WORK_DIR),
        '--events_per_pixel', str(EVENTS_PER_PIXEL),
        '--start_s',          str(start_s),
        '--compress_jpeg',
    ]
    if SMOKE_EVENTS:
        cmd += ['--max_events', str(SMOKE_EVENTS)]

    rc = run_streaming(cmd)
    if rc != 0:
        log(f'ERROR: reconstruct.py failed for {seq} (exit code {rc})')
        raise RuntimeError(f'reconstruct.py failed for {seq} (exit code {rc})')

log('=== Reconstruction done ===')

## 6 · Train YOLO

In [ ]:
if SKIP_TRAINING:
    log('SKIP_TRAINING=True — skipping training')
else:
    log('=== Training started ===')

    resume_yaml = DATASET_DIR / 'dataset.yaml'
    resume_yaml.parent.mkdir(parents=True, exist_ok=True)
    resume_yaml.write_text(f"""path: {DATASET_DIR}
train: train.txt
val:   val.txt

nc: 1
names:
  0: drone
""")
    log(f'dataset.yaml pre-written → {DATASET_DIR}')

    cmd = [
        sys.executable, str(LOCAL_SCRIPTS / 'train_yolo.py'),
        '--sequences',     *SEQUENCES,
        '--val_sequences', *VAL_SEQUENCES,
        '--raw_root',      str(RAW_ROOT),
        '--recon_root',    str(RECON_ROOT),
        '--out_dir',       str(DATASET_DIR),
        '--runs_dir',      str(RUNS_DIR),
        '--weights',       str(WEIGHTS_OUT),
        '--model',         MODEL,
        '--epochs',        str(EPOCHS),
        '--batch',         str(BATCH),
    ]

    if RESUME:
        cmd += ['--resume']

    rc = run_streaming(cmd)
    if rc != 0:
        log(f'ERROR: train_yolo.py failed (exit code {rc})')
        raise RuntimeError('train_yolo.py failed')

    log(f'=== Training done — weights at {WEIGHTS_OUT} ===')

## 7 · Zip new reconstruction frames for GUI download

Bundles newly reconstructed frames (sequences 44–47, 146) into a single zip.
Sequences 84/85/201/124 are already in `fred-frames-all` and don't need re-downloading.

In [ ]:
import zipfile
from pathlib import Path

zip_path = AMI_WORK / 'frames_e2vid_run6.zip'
log('=== Zipping reconstruction frames (stream-and-delete) ===')
frame_count = 0
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as zf:
    for seq in ZIP_SEQUENCES:
        recon_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if not recon_dir.exists():
            log(f'  {seq}: no frames found, skipping')
            continue
        ts_file = recon_dir / 'timestamps.txt'
        if ts_file.exists():
            zf.write(ts_file, ts_file.relative_to(AMI_WORK))
        frames = sorted(recon_dir.glob('frame_*.jpg'))
        for frame in frames:
            zf.write(frame, frame.relative_to(AMI_WORK))
            frame.unlink()  # delete source immediately — keeps disk flat
            frame_count += 1
        log(f'  {seq}: {len(frames)} frames added')

size_mb = zip_path.stat().st_size / 1e6
log(f'frames_e2vid_run6.zip: {frame_count} frames, {size_mb:.0f} MB → {zip_path}')


In [ ]:
# Free disk space so outputs fit in the Kaggle snapshot.
import shutil, os

freed = 0

def _rmdir(p):
    global freed
    p = Path(p)
    if p.exists():
        size = sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
        shutil.rmtree(p)
        freed += size
        log(f'  deleted {p}  ({size / 1e9:.2f} GB)')

def _rm(p):
    global freed
    p = Path(p)
    if p.is_file():
        freed += p.stat().st_size
        p.unlink()

# ── Save YOLO outputs to kpis/ before cleanup ────────────────────────────────
e2vid_run = RUNS_DIR / 'e2vid'
kpis_dir  = AMI_WORK / 'kpis'
kpis_dir.mkdir(parents=True, exist_ok=True)

KEEP = {
    'results.csv', 'results.png',
    'confusion_matrix.png', 'confusion_matrix_normalized.png',
    'BoxF1_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'BoxPR_curve.png',
    'labels.jpg', 'labels_correlogram.jpg',
    'train_batch0.jpg', 'train_batch1.jpg', 'train_batch2.jpg',
    'val_batch0_labels.jpg', 'val_batch0_pred.jpg',
    'val_batch1_labels.jpg', 'val_batch1_pred.jpg',
    'val_batch2_labels.jpg', 'val_batch2_pred.jpg',
}

if e2vid_run.exists():
    for p in e2vid_run.iterdir():
        if p.is_file() and p.name in KEEP:
            shutil.copy(p, kpis_dir / p.name)
            log(f'  saved {p.name} → kpis/')

# ── Delete large directories ──────────────────────────────────────────────────
_rmdir(RECON_ROOT.parent)

if e2vid_run.exists():
    for p in list(e2vid_run.iterdir()):
        if p.name != 'weights':
            _rmdir(p) if p.is_dir() else _rm(p)

_rmdir(WORK_DIR)

log(f'Freed {freed / 1e9:.2f} GB total')
os.system('df -h /kaggle/working')

## 8 · Download results

After the notebook finishes, Kaggle saves `/kaggle/working/` as the run output.

**Weights + KPIs + logs (fast — ~50 MB):**
```bash
bash scripts/sync_from_kaggle.sh
```

**New reconstruction frames for GUI (sequences 44–47, 146):**
```bash
bash scripts/sync_from_kaggle.sh --frames-zip
```
Extracts `frames_e2vid_run6.zip` into `data/processed/`.

Then rebuild and push the service:
```bash
cp data/yolo_runs/e2vid/weights/best.pt services/e2vid/weights/yolo_e2vid.pt
docker build -t ghcr.io/gennepy/ami-e2vid:latest services/e2vid/
gh auth token | docker login ghcr.io -u gennepy --password-stdin
docker push ghcr.io/gennepy/ami-e2vid:latest
```